# Day 16 — Detection Baseline

## Objective

To train a transfer-learning YOLO object detector on the original raw dataset, evaluate detection performance using Precision,Recall,mAP@0.5 and mAP@0.5:0.95,analyse class-wise errors,measure model size and inference time,and save the trained model and prediction outputs.

## Dataset Scope

The detector will be trained and evaluated separately on:

- DIATAquarium
- Aquatic Plant
- Well

## Baseline

Original Images → YOLO Object Detection → Detection Results → Evaluation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
DATASET_V1="/content/drive/MyDrive/Dataset_V1"
LOCAL_DATASET="/content/Dataset_V1"
DATASETS=[
    "DIATAquarium.v4i.yolov8",
    "Aquatic Plant.v2i.yolov8",
    "well.v8i.yolov8"
]

print("Google Drive dataset:",os.path.exists(DATASET_V1))

Google Drive dataset: True


In [ ]:
import os
import shutil
DATASET_V1="/content/drive/MyDrive/Dataset_V1"
LOCAL_DATASET="/content/Dataset_V1"
if os.path.exists(LOCAL_DATASET):
    print("Local Dataset V1 already exists")
else:
    print("Copying Dataset V1 to Colab local storage...")
    shutil.copytree(DATASET_V1,LOCAL_DATASET)
    print("Copy completed")
print("Local dataset exists:",os.path.exists(LOCAL_DATASET))

Copying Dataset V1 to Colab local storage...
Copy completed
Local dataset exists: True


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 8.1 MB/s eta 0:00:00


In [ ]:
for dataset in DATASETS:
    path=os.path.join(LOCAL_DATASET,dataset)
    print(dataset,"->",os.path.exists(path))

DIATAquarium.v4i.yolov8 -> True
Aquatic Plant.v2i.yolov8 -> True
well.v8i.yolov8 -> True


In [ ]:
import os
import yaml
LOCAL_DATASET="/content/Dataset_V1"
DATASET_INFO={
    "DIATAquarium.v4i.yolov8":[
        "Aquatic Plant",
        "Camera",
        "Conch-Shell",
        "Fish",
        "Fishes",
        "Pluco",
        "Shark",
        "Temperature Sensor",
        "Water-Pump",
        "Water-filter",
        "betta"
    ],
    "Aquatic Plant.v2i.yolov8":[
        "Aquatic_plant"
    ],
    "well.v8i.yolov8":[
        "Inlet-pipe",
        "fishes",
        "school-of-fish",
        "stone"
    ]
}
for dataset,names in DATASET_INFO.items():
    dataset_path=os.path.join(LOCAL_DATASET,dataset)
    yaml_path=os.path.join(dataset_path,"baseline.yaml")
    config={
        "path":dataset_path,
        "train":"train/images",
        "val":"valid/images",
        "nc":len(names),
        "names":names
    }
    with open(yaml_path,"w") as f:
        yaml.dump(config,f,sort_keys=False)
    print("Created:",yaml_path)

Created: /content/Dataset_V1/DIATAquarium.v4i.yolov8/baseline.yaml
Created: /content/Dataset_V1/Aquatic Plant.v2i.yolov8/baseline.yaml
Created: /content/Dataset_V1/well.v8i.yolov8/baseline.yaml


In [ ]:
from ultralytics import YOLO
model=YOLO("yolov8n.pt")
print("YOLOv8n pretrained model loaded successfully")

YOLOv8n pretrained model loaded successfully


In [ ]:
from ultralytics import YOLO
model=YOLO("yolov8n.pt")
DIAT_YAML="/content/Dataset_V1/DIATAquarium.v4i.yolov8/baseline.yaml"
results=model.train(
    data=DIAT_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/Day16_YOLO_Baseline",
    name="DIATAquarium"
)

New https://pypi.org/project/ultralytics/8.4.154 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.153 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1/DIATAquarium.v4i.yolov8/baseline.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

In [ ]:
import shutil
import os
DRIVE_RESULTS="/content/drive/MyDrive/Day16_YOLO_Baseline"
os.makedirs(DRIVE_RESULTS,exist_ok=True)
shutil.copytree(
    "/content/Day16_YOLO_Baseline/DIATAquarium",
    os.path.join(DRIVE_RESULTS,"DIATAquarium"),
    dirs_exist_ok=True
)
print("DIAT baseline results saved to Google Drive")

DIAT baseline results saved to Google Drive


In [ ]:
RESULTS_PATH="/content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium"
print("DIAT results folder:",os.path.exists(RESULTS_PATH))
if os.path.exists(RESULTS_PATH):
    print("\nSaved files/folders:")
    for item in os.listdir(RESULTS_PATH):
        print(item)

DIAT results folder: True

Saved files/folders:
weights
labels.jpg
train_batch0.jpg
train_batch1.jpg
train_batch2.jpg
train_batch10161.jpg
train_batch10160.jpg
train_batch10162.jpg
results.csv
val_batch0_labels.jpg
val_batch1_labels.jpg
val_batch0_pred.jpg
val_batch1_pred.jpg
val_batch2_labels.jpg
val_batch2_pred.jpg
BoxPR_curve.png
BoxF1_curve.png
BoxP_curve.png
BoxR_curve.png
confusion_matrix_normalized.png
confusion_matrix.png
results.png
args.yaml


In [ ]:
import pandas as pd
RESULTS_CSV="/content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/results.csv"
df=pd.read_csv(RESULTS_CSV)
print("Available columns:")
print(df.columns.tolist())
print("\nLast epoch results:")
print(df.iloc[-1])

Available columns:
['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']

Last epoch results:
epoch                     30.000000
time                    6027.170000
train/box_loss             0.642700
train/cls_loss             0.363420
train/dfl_loss             0.954090
metrics/precision(B)       0.900170
metrics/recall(B)          0.920860
metrics/mAP50(B)           0.927230
metrics/mAP50-95(B)        0.659000
val/box_loss               0.950560
val/cls_loss               0.483550
val/dfl_loss               1.062320
lr/pg0                     0.000029
lr/pg1                     0.000029
lr/pg2                     0.000029
Name: 29, dtype: float64


In [ ]:
import pandas as pd
RESULTS_CSV="/content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/results.csv"
df=pd.read_csv(RESULTS_CSV)
last=df.iloc[-1]
print("DIATAquarium - YOLOv8n Original Baseline")
print("-----------------------------------------")
print("Epoch:",int(last["epoch"]))
print("Precision:",round(last["metrics/precision(B)"]*100,2),"%")
print("Recall:",round(last["metrics/recall(B)"]*100,2),"%")
print("mAP@0.5:",round(last["metrics/mAP50(B)"]*100,2),"%")
print("mAP@0.5:0.95:",round(last["metrics/mAP50-95(B)"]*100,2),"%")

DIATAquarium - YOLOv8n Original Baseline
-----------------------------------------
Epoch: 30
Precision: 90.02 %
Recall: 92.09 %
mAP@0.5: 92.72 %
mAP@0.5:0.95: 65.9 %


In [ ]:
from ultralytics import YOLO
MODEL_PATH="/content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/weights/best.pt"
DATASET_PATH="/content/Dataset_V1/DIATAquarium.v4i.yolov8/baseline.yaml"
model=YOLO(MODEL_PATH)
results=model.val(
    data=DATASET_PATH,
    imgsz=640,
    batch=16,
    device=0,
    verbose=False
)
names=model.names
print("DIATAquarium - Class-wise Results")
print("---------------------------------")
for i,name in names.items():
    print(name,
        "| Precision:",round(results.box.p[i]*100,2),"%",
        "| Recall:",round(results.box.r[i]*100,2),"%",
        "| mAP@0.5:",round(results.box.ap50[i]*100,2),"%",
        "| mAP@0.5:0.95:",round(results.box.ap[i]*100,2),"%"
    )

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,793 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.3 ms, read: 39.5±18.5 MB/s, size: 140.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Dataset_V1/DIATAquarium.v4i.yolov8/valid/labels... 773 images, 59 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 773/773 111.8it/s 6.9s
val: New cache created: /content/drive/MyDrive/Dataset_V1/DIATAquarium.v4i.yolov8/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 3.4it/s 14.6s
                   all        773       1504        0.9      0.921      0.927      0.659
Speed: 1.5ms preprocess, 3.4ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /content/runs/

In [ ]:
import os
from ultralytics import YOLO
MODEL_PATH="/content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/weights/best.pt"
model=YOLO(MODEL_PATH)
model_size=os.path.getsize(MODEL_PATH)/(1024*1024)
print("DIATAquarium - YOLOv8n Model Information")
print("------------------------------------------")
print("Model: YOLOv8n")
print("Model size:",round(model_size,2),"MB")
print("Parameters:",model.model.model[-1].nc)

DIATAquarium - YOLOv8n Model Information
------------------------------------------
Model: YOLOv8n
Model size: 5.95 MB
Parameters: 11


In [ ]:
import time
import cv2
import os
from ultralytics import YOLO
MODEL_PATH="/content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/weights/best.pt"
IMAGE_DIR="/content/Dataset_V1/DIATAquarium.v4i.yolov8/valid/images"
model=YOLO(MODEL_PATH)
images=[os.path.join(IMAGE_DIR,f) for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith((".jpg",".jpeg",".png"))]
images=images[:100]
start=time.time()
for path in images:
    img=cv2.imread(path)
    model.predict(img,imgsz=640,device=0,verbose=False)
end=time.time()
total_time=end-start
avg_time=(total_time/len(images))*1000
print("DIATAquarium - YOLOv8n Inference Time")
print("--------------------------------------")
print("Images tested:",len(images))
print("Total time:",round(total_time,2),"seconds")
print("Average inference time:",round(avg_time,2),"ms/image")
print("Approx FPS:",round(1000/avg_time,2))

DIATAquarium - YOLOv8n Inference Time
--------------------------------------
Images tested: 100
Total time: 2.13 seconds
Average inference time: 21.25 ms/image
Approx FPS: 47.05


In [ ]:
from ultralytics import YOLO
import os
MODEL_PATH="/content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/weights/best.pt"
IMAGE_DIR="/content/Dataset_V1/DIATAquarium.v4i.yolov8/valid/images"
OUTPUT_DIR="/content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/predictions"
model=YOLO(MODEL_PATH)
images=[
    os.path.join(IMAGE_DIR,f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg",".jpeg",".png"))
][:10]
model.predict(source=images,
    imgsz=640,
    device=0,
    save=True,
    project=OUTPUT_DIR,
    name="validation_predictions",
    verbose=False
)
print("Prediction images saved.")
print("Location:",os.path.join(OUTPUT_DIR,"validation_predictions"))

Results saved to /content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/predictions/validation_predictions
Prediction images saved.
Location: /content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/predictions/validation_predictions


# DIATAquarium — YOLOv8n Original Baseline

## Dataset
- Training images: 8,121
- Validation images: 773
- Test set: Not available in the supplied local dataset

## Model Configuration
- Model: YOLOv8n
- Epochs: 30
- Image Size: 640 × 640
- Batch Size: 16
- Enhancement: None (Original Images)

## Validation Performance
| Metric | Result |
|---|---:|
| Precision | 90.02% |
| Recall | 92.09% |
| mAP@0.5 | 92.72% |
| mAP@0.5:0.95 | 65.90% |

## Class-wise Evaluation
Class-wise Precision, Recall, mAP@0.5 and mAP@0.5:0.95 were evaluated for all 11 classes.

## Model Information
- Model size: 5.95 MB
- Parameters: 3,007,793
- GFLOPs: 8.1

## Inference Performance
- Average measured inference time: 21.25 ms/image
- Approximate FPS: 47.05

## Saved Outputs
- Best model: `/content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/weights/best.pt`
- Prediction outputs: `/content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/predictions/validation_predictions`

## Evaluation Note
The reported Precision, Recall and mAP values are validation metrics obtained using the 773-image validation set. A separate local test set was not available for DIATAquarium. Source-level overlap across dataset splits was identified during dataset inspection and is documented as a limitation.

#Aquatic plant

In [ ]:
import os
AP_PATH="/content/Dataset_V1/Aquatic Plant.v2i.yolov8"
yaml_content=f"""path: {AP_PATH}
train: train/images
val: valid/images
test: test/images

nc: 1
names:
  0: Aquatic_plant
"""
YAML_PATH=os.path.join(AP_PATH,"baseline.yaml")
with open(YAML_PATH,"w") as f:
    f.write(yaml_content)
print("YAML created:",YAML_PATH)
with open(YAML_PATH,"r") as f:
    print(f.read())

YAML created: /content/Dataset_V1/Aquatic Plant.v2i.yolov8/baseline.yaml
path: /content/Dataset_V1/Aquatic Plant.v2i.yolov8
train: train/images
val: valid/images
test: test/images

nc: 1
names:
  0: Aquatic_plant



In [ ]:
from ultralytics import YOLO
AP_YAML="/content/Dataset_V1/Aquatic Plant.v2i.yolov8/baseline.yaml"
model=YOLO("yolov8n.pt")
results=model.train(
    data=AP_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/Day16_YOLO_Baseline",
    name="Aquatic_Plant"
)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1/Aquatic Plant.v2i.yolov8/baseline.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Aquatic_Plant, nb

In [ ]:
from ultralytics import YOLO
MODEL_PATH="/content/Day16_YOLO_Baseline/Aquatic_Plant/weights/best.pt"
DATASET_PATH="/content/Dataset_V1/Aquatic Plant.v2i.yolov8/baseline.yaml"
model=YOLO(MODEL_PATH)
results=model.val(
    data=DATASET_PATH,
    imgsz=640,
    batch=16,
    device=0,
    verbose=False
)
names=model.names
print("Aquatic Plant - Class-wise Results")
print("----------------------------------")
for i,name in names.items():
    print(
        name,
        "| Precision:",round(results.box.p[i]*100,2),"%",
        "| Recall:",round(results.box.r[i]*100,2),"%",
        "| mAP@0.5:",round(results.box.ap50[i]*100,2),"%",
        "| mAP@0.5:0.95:",round(results.box.ap[i]*100,2),"%"
    )

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 668.1±155.7 MB/s, size: 12.9 KB)
val: Scanning /content/Dataset_V1/Aquatic Plant.v2i.yolov8/valid/labels.cache... 179 images, 79 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 179/179 46.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 2.7it/s 4.4s
                   all        179        108      0.946      0.935       0.98      0.708
Speed: 5.9ms preprocess, 6.2ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to /content/runs/detect/val-3
Aquatic Plant - Class-wise Results
----------------------------------
Aquatic_plant | Precision: 94.64 % | Recall: 93.52 % | mAP@0.5: 98.05 % | mAP@0.5:0.95: 70.79 %


In [ ]:
import os
MODEL_PATH="/content/Day16_YOLO_Baseline/Aquatic_Plant/weights/best.pt"
size_mb=os.path.getsize(MODEL_PATH)/(1024*1024)
print("Aquatic Plant YOLOv8n")
print("--------------------")
print("Model size:",round(size_mb,2),"MB")

Aquatic Plant YOLOv8n
--------------------
Model size: 5.96 MB


In [ ]:
import os
import shutil
SOURCE="/content/Day16_YOLO_Baseline/Aquatic_Plant/weights/best.pt"
DEST="/content/drive/MyDrive/Day16_YOLO_Baseline/Aquatic_Plant/weights"
os.makedirs(DEST,exist_ok=True)
shutil.copy2(SOURCE,os.path.join(DEST,"Aquatic_Plant_best.pt"))
print("Aquatic Plant model saved successfully!")
print("Location:",os.path.join(DEST,"Aquatic_Plant_best.pt"))
print("Size:",round(os.path.getsize(os.path.join(DEST,"Aquatic_Plant_best.pt"))/(1024*1024),2),"MB")

Aquatic Plant model saved successfully!
Location: /content/drive/MyDrive/Day16_YOLO_Baseline/Aquatic_Plant/weights/Aquatic_Plant_best.pt
Size: 5.96 MB


In [ ]:
from ultralytics import YOLO
import os
import time
MODEL_PATH="/content/Day16_YOLO_Baseline/Aquatic_Plant/weights/best.pt"
IMAGE_DIR="/content/Dataset_V1/Aquatic Plant.v2i.yolov8/valid/images"
model=YOLO(MODEL_PATH)
images=[
    os.path.join(IMAGE_DIR,f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg",".jpeg",".png"))
][:100]
start=time.time()
model.predict(
    source=images,
    imgsz=640,
    device=0,
    verbose=False
)
total_time=time.time()-start
avg_time=total_time/len(images)
fps=1/avg_time

print("Aquatic Plant - Inference Time")
print("------------------------------")
print("Images:",len(images))
print("Total time:",round(total_time,2),"seconds")
print("Average time per image:",round(avg_time*1000,2),"ms")
print("Approximate FPS:",round(fps,2))

Aquatic Plant - Inference Time
------------------------------
Images: 100
Total time: 1.26 seconds
Average time per image: 12.6 ms
Approximate FPS: 79.35


In [ ]:
from ultralytics import YOLO
import os
MODEL_PATH="/content/Day16_YOLO_Baseline/Aquatic_Plant/weights/best.pt"
IMAGE_DIR="/content/Dataset_V1/Aquatic Plant.v2i.yolov8/valid/images"
OUTPUT_DIR="/content/drive/MyDrive/Day16_YOLO_Baseline/Aquatic_Plant/predictions"
os.makedirs(OUTPUT_DIR,exist_ok=True)
model=YOLO(MODEL_PATH)
images=[
    os.path.join(IMAGE_DIR,f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg",".jpeg",".png"))
][:10]
model.predict(
    source=images,
    imgsz=640,
    device=0,
    save=True,
    project=OUTPUT_DIR,
    name="validation_predictions",
    verbose=False
)
print("Prediction images saved successfully!")
print("Location:",os.path.join(OUTPUT_DIR,"validation_predictions"))

Results saved to /content/drive/MyDrive/Day16_YOLO_Baseline/Aquatic_Plant/predictions/validation_predictions
Prediction images saved successfully!
Location: /content/drive/MyDrive/Day16_YOLO_Baseline/Aquatic_Plant/predictions/validation_predictions


In [ ]:
from ultralytics import YOLO
MODEL_PATH="/content/Day16_YOLO_Baseline/Aquatic_Plant/weights/best.pt"
DATASET_PATH="/content/Dataset_V1/Aquatic Plant.v2i.yolov8/baseline.yaml"
model=YOLO(MODEL_PATH)
results=model.val(
    data=DATASET_PATH,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    verbose=False
)
print("Aquatic Plant - Test Set Results")
print("--------------------------------")
print("Precision:",round(results.box.mp*100,2),"%")
print("Recall:",round(results.box.mr*100,2),"%")
print("mAP@0.5:",round(results.box.map50*100,2),"%")
print("mAP@0.5:0.95:",round(results.box.map*100,2),"%")

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 52.1±91.9 MB/s, size: 16.7 KB)
val: Scanning /content/Dataset_V1/Aquatic Plant.v2i.yolov8/test/labels... 89 images, 38 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 89/89 758.2it/s 0.1s
val: New cache created: /content/Dataset_V1/Aquatic Plant.v2i.yolov8/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.3it/s 2.6s
                   all         89         54      0.957      0.944      0.985      0.706
Speed: 6.4ms preprocess, 7.3ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /content/runs/detect/val-4
Aquatic Plant - Test Set Results
--------------------------------
Precision: 95.69 %
Recall: 94.44 %
mAP@0.5: 98.55 %
mAP@0.5:0.95: 70.63 %


# Aquatic Plant — YOLOv8n Original Baseline

## Dataset
- Training images: 1,892
- Validation images: 179
- Test images: 89
- Number of classes: 1
- Class: Aquatic_plant

## Model Configuration
- Model: YOLOv8n
- Epochs: 30
- Image Size: 640 × 640
- Batch Size: 16
- Device: Tesla T4 GPU
- Enhancement: None (Original Images)

## Validation Performance
| Metric | Result |
|---|---:|
| Precision | 94.64% |
| Recall | 93.52% |
| mAP@0.5 | 98.05% |
| mAP@0.5:0.95 | 70.79% |

## Class-wise Evaluation
| Class | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 |
|---|---:|---:|---:|---:|
| Aquatic_plant | 94.64% | 93.52% | 98.05% | 70.79% |

## Test Set Performance
- Test images: 89
- Test instances: 54

| Metric | Result |
|---|---:|
| Precision | 95.69% |
| Recall | 94.44% |
| mAP@0.5 | 98.55% |
| mAP@0.5:0.95 | 70.63% |

## Model Information
- Model size: 5.96 MB
- Parameters: 3,005,843
- GFLOPs: 8.1

## Validation Speed
- Preprocess: 5.9 ms/image
- Inference: 6.2 ms/image
- Postprocess: 2.6 ms/image

## Measured Prediction Performance
- Images measured: 100
- Total prediction time: 1.26 seconds
- Average prediction time: 12.6 ms/image
- Approximate FPS: 79.35

## Saved Outputs
- Best model: `/content/drive/MyDrive/Day16_YOLO_Baseline/Aquatic_Plant/weights/Aquatic_Plant_best.pt`
- Prediction outputs: `/content/drive/MyDrive/Day16_YOLO_Baseline/Aquatic_Plant/predictions/validation_predictions`

## Evaluation Note
The validation results were obtained using the 179-image validation set. The final test results were obtained separately using the 89-image test set. The saved YOLOv8n model was not retrained during test evaluation.

#Well training

In [ ]:
WELL_YAML="/content/Dataset_V1/well.v8i.yolov8/baseline.yaml"
with open(WELL_YAML,"w") as f:
    f.write("""path: /content/Dataset_V1/well.v8i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 4
names:
  0: Inlet-pipe
  1: fishes
  2: school-of-fish
  3: stone
""")
print("Well baseline.yaml created successfully!")
print(WELL_YAML)

Well baseline.yaml created successfully!
/content/Dataset_V1/well.v8i.yolov8/baseline.yaml


In [ ]:
from ultralytics import YOLO
WELL_YAML="/content/Dataset_V1/well.v8i.yolov8/baseline.yaml"
model=YOLO("yolov8n.pt")
results=model.train(
    data=WELL_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/Day16_YOLO_Baseline",
    name="Well"
)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1/well.v8i.yolov8/baseline.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Well, nbs=64, nms=None, op

In [ ]:
import os
import shutil
SOURCE="/content/Day16_YOLO_Baseline/Well/weights/best.pt"
DEST="/content/drive/MyDrive/Day16_YOLO_Baseline/Well/weights"
os.makedirs(DEST,exist_ok=True)
shutil.copy2(SOURCE,os.path.join(DEST,"Well_best.pt"))
print("Well model saved successfully!")
print("Location:",os.path.join(DEST,"Well_best.pt"))
print("Size:",round(os.path.getsize(os.path.join(DEST,"Well_best.pt"))/(1024*1024),2),"MB")

Well model saved successfully!
Location: /content/drive/MyDrive/Day16_YOLO_Baseline/Well/weights/Well_best.pt
Size: 5.96 MB


In [ ]:
print("Class names:",model.names)
print("Number of metric entries:",len(results.box.p))
print("Class indices with metrics:",list(range(len(results.box.p))))

Class names: {0: 'Inlet-pipe', 1: 'fishes', 2: 'school-of-fish', 3: 'stone'}
Number of metric entries: 3
Class indices with metrics: [0, 1, 2]


In [ ]:
from ultralytics import YOLO
MODEL_PATH="/content/Day16_YOLO_Baseline/Well/weights/best.pt"
DATASET_PATH="/content/Dataset_V1/well.v8i.yolov8/baseline.yaml"
model=YOLO(MODEL_PATH)
results=model.val(
    data=DATASET_PATH,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    verbose=True
)
print("Well Test Set Results")
print("---------------------")
print("Precision:",round(results.box.mp*100,2),"%")
print("Recall:",round(results.box.mr*100,2),"%")
print("mAP@0.5:",round(results.box.map50*100,2),"%")
print("mAP@0.5:0.95:",round(results.box.map*100,2),"%")

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 12.3±3.9 MB/s, size: 20.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/Dataset_V1/well.v8i.yolov8/test/labels... 184 images, 21 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 184/184 745.9it/s 0.2s
val: New cache created: /content/Dataset_V1/well.v8i.yolov8/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.6it/s 7.4s
                   all        184       2111      0.446      0.383      0.422      0.185
                fishes        161       2065      0.671      0.731      0.713      0.272
        school-of-fish         23         43      0.668      0.419      0

In [ ]:
import os
import time
from ultralytics import YOLO
MODEL_PATH="/content/Day16_YOLO_Baseline/Well/weights/best.pt"
IMAGE_DIR="/content/Dataset_V1/well.v8i.yolov8/valid/images"
model=YOLO(MODEL_PATH)
images=[
    os.path.join(IMAGE_DIR,f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg",".jpeg",".png"))
][:100]
start=time.time()
for image in images:
    model.predict(
        source=image,
        imgsz=640,
        device=0,
        verbose=False
    )
total_time=time.time()-start
avg_time=total_time/len(images)
fps=1/avg_time
print("Well Prediction Performance")
print("---------------------------")
print("Images measured:",len(images))
print("Total prediction time:",round(total_time,2),"seconds")
print("Average prediction time:",round(avg_time*1000,2),"ms/image")
print("Approximate FPS:",round(fps,2))

Well Prediction Performance
---------------------------
Images measured: 100
Total prediction time: 1.85 seconds
Average prediction time: 18.55 ms/image
Approximate FPS: 53.91


In [ ]:
import os
from ultralytics import YOLO
MODEL_PATH="/content/Day16_YOLO_Baseline/Well/weights/best.pt"
IMAGE_DIR="/content/Dataset_V1/well.v8i.yolov8/valid/images"
SAVE_DIR="/content/drive/MyDrive/Day16_YOLO_Baseline/Well/predictions/visual_outputs"
os.makedirs(SAVE_DIR,exist_ok=True)
model=YOLO(MODEL_PATH)

images=[
    os.path.join(IMAGE_DIR,f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg",".jpeg",".png"))
][:10]
results=model.predict(
    source=images,
    imgsz=640,
    device=0,
    save=True,
    project=SAVE_DIR,
    name="well_validation_predictions",
    exist_ok=True,
    verbose=False
)
print("Well prediction images saved successfully!")
print("Location:",os.path.join(SAVE_DIR,"well_validation_predictions"))
print("Images saved:",len(images))

Results saved to /content/drive/MyDrive/Day16_YOLO_Baseline/Well/predictions/visual_outputs/well_validation_predictions
Well prediction images saved successfully!
Location: /content/drive/MyDrive/Day16_YOLO_Baseline/Well/predictions/visual_outputs/well_validation_predictions
Images saved: 10


In [ ]:
SUMMARY_PATH="/content/drive/MyDrive/Day16_YOLO_Baseline/Well/Well_Day16_Summary.md"
content="""# Day 16 — Well YOLOv8n Original Baseline
## Dataset
- Dataset: Well
- Training images: 3,795
- Validation images: 191
- Test images: 184
- Number of classes: 4
- Classes: Inlet-pipe, fishes, school-of-fish, stone
## Model Configuration
- Model: YOLOv8n
- Transfer learning: Yes
- Epochs: 30
- Image Size: 640 × 640
- Batch Size: 16
- Device: Tesla T4 GPU
- Enhancement: None (Original Images)
## Validation Performance
| Metric | Result |
|---|---:|
| Precision | 35.60% |
| Recall | 36.80% |
| mAP@0.5 | 36.50% |
| mAP@0.5:0.95 | 16.40% |
## Class-wise Validation Performance
| Class | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 |
|---|---:|---:|---:|---:|
| Inlet-pipe | 64.16% | 74.80% | 68.99% | 26.62% |
| fishes | 42.72% | 35.48% | 40.59% | 22.60% |
| school-of-fish | 0.00% | 0.00% | 0.00% | 0.00% |
| stone | — | — | — | — |
Note: The validation output returned metrics for three classes. No metric entry was returned for stone, so no value is assigned.
## Test Set Performance
- Test images: 184
- Test instances: 2,111
- Background images: 21
- Corrupt images: 0

| Metric | Result |
|---|---:|
| Precision | 44.64% |
| Recall | 38.31% |
| mAP@0.5 | 42.20% |
| mAP@0.5:0.95 | 18.46% |
## Class-wise Test Performance
| Class | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 |
|---|---:|---:|---:|---:|
| fishes | 67.10% | 73.10% | 71.30% | 27.20% |
| school-of-fish | 66.80% | 41.90% | 55.30% | 28.10% |
| stone | 0.00% | 0.00% | 0.00% | 0.00% |
| Inlet-pipe | — | — | — | — |
Note: Inlet-pipe did not appear in the test metrics output, so no value is assigned.
## Model Information
- Model size: approximately 5.96 MB
- Parameters: approximately 3.01 million
- GFLOPs: approximately 8.1
## Validation Speed
- Preprocess: 8.5 ms/image
- Inference: 5.8 ms/image
- Postprocess: 5.5 ms/image
## Measured Prediction Performance
- Images measured: 100
- Total prediction time: 1.85 seconds
- Average prediction time: 18.55 ms/image
- Approximate FPS: 53.91
## Saved Outputs
### Best Model
/content/drive/MyDrive/Day16_YOLO_Baseline/Well/weights/Well_best.pt

### Prediction Images
/content/drive/MyDrive/Day16_YOLO_Baseline/Well/predictions/visual_outputs/well_validation_predictions

- Number of prediction images saved: 10

## Evaluation Note
The validation results were obtained using the 191-image validation set. The test results were obtained separately using the 184-image test set. The saved YOLOv8n model was not retrained during test evaluation.

Source-level overlap across dataset splits was identified during dataset inspection and is documented as a project limitation.

## Day 16 Status
Well YOLOv8n original baseline completed successfully.
"""

with open(SUMMARY_PATH,"w") as f:
    f.write(content)

print("Well Day 16 summary created successfully!")
print("Location:",SUMMARY_PATH)

Well Day 16 summary created successfully!
Location: /content/drive/MyDrive/Day16_YOLO_Baseline/Well/Well_Day16_Summary.md
